# 09 | Reproduzir as figuras quantitativas

**Papel nesta entrega:** Principal: codigo grafico documentado da entrega anterior

**Origem:** Entrega de 4 out. 2026; 05_figuras_e_tabelas_finais.ipynb

Copia de publicacao organizada em 7 out. 2026. Saidas, contadores e metadados de sessao foram removidos; isso nao significa que a campanha original nao tenha sido executada. Nenhum resultado novo foi produzido na curadoria. O registro de proveniencia relaciona codigo, fonte e alteracoes de configuracao.

Recalcula graficos a partir das tabelas congeladas; nao treina modelos. A numeração foi mapeada para Figuras 2-6, e o painel arquivado de microscopia corresponde a Figura 7. A Figura 1 real e as oito tabelas da versao aprovada estao no caderno 10. A aparencia de uma nova renderizacao pode variar com bibliotecas; os pixels aprovados estao arquivados separadamente.

# 05 | Figuras e tabelas finais

Este caderno refaz as figuras quantitativas com as tabelas congeladas da entrega. Não treina modelos, não reabre R4 e não procura candidatos. A imagem de microscopia é copiada do painel real arquivado, nunca sintetizada. As saídas ficam em uma nova pasta e não substituem as figuras do manuscrito.

**Unidades:** condição experimental nas comparações; predição externa na regressão; perfil do poço na integração visual. Esses denominadores não são intercambiáveis.

## 1. Abrir as tabelas e preparar uma pasta nova

Os arquivos vêm no pacote completo. Não instale novamente as bibliotecas da modelagem apenas para produzir estas figuras.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive

drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/mba_tcc/TCC_CellPainting_cadernos_repositorio/04_fontes_e_referencia/recursos')
if not (BASE / 'MANIFESTO_SHA256.json').is_file():
    raise FileNotFoundError('Execute primeiro o caderno 00 para instalar o pacote completo.')
T = BASE / 'tabelas/comparacao'
A = BASE / 'tabelas/alda1'
SITES = ['IMTM', 'FMP', 'MEDINA', 'USC']
EXECUCAO = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
F = BASE.parent / 'resultados_edicao_final' / ('figuras_' + EXECUCAO)
F.mkdir(parents=True, exist_ok=False)
EXCEDENCIAS = pd.read_csv(T / 'excedencia_blocos_completos.csv')
CAP = {}
plt.rcParams.update({'font.family': 'DejaVu Sans', 'font.size': 10, 'axes.titlesize': 12})

def save(fig, name, caption):
    """Salva PNG para leitura e SVG vetorial; nao altera valores das tabelas."""
    fig.tight_layout()
    fig.savefig(F / (name + '.png'), dpi=220, bbox_inches='tight')
    fig.savefig(F / (name + '.svg'), bbox_inches='tight')
    plt.close(fig)
    CAP[name] = caption
    print('Figura salva:', name)


## 2. Excedência por bloco

A mesma regra A > 1 é contada em pelo menos duas réplicas. Cada local conserva seu denominador de condições completas. Exceder apenas Mito não equivale a atender à prioridade completa.

In [ ]:
bl = ['AGP', 'DNA', 'ER', 'Mito', 'Morfologia_Cells', 'Morfologia_Cyto', 'Morfologia_Nuc']
values = []
for site in SITES:
    d = EXCEDENCIAS[EXCEDENCIAS.site.eq(site)].set_index('bloco')
    values.append([100 * d.loc[b, 'n_em_pelo_menos_2'] / d.loc[b, 'n_condicoes'] for b in bl])
fig, ax = plt.subplots(figsize=(8.1, 3.8))
im = ax.imshow(np.array(values).T, aspect='auto', vmin=0, vmax=100)
ax.set_xticks(range(4), SITES)
ax.set_yticks(range(7), ['AGP', 'DNA', 'ER', 'Mito', 'Forma: células', 'Forma: citoplasma', 'Forma: núcleos'])
for i in range(7):
    for j in range(4):
        ax.text(j, i, f'{values[j][i]:.1f}%', ha='center', va='center', bbox={'facecolor': 'white', 'alpha': 0.7, 'edgecolor': 'none', 'pad': 1})
fig.colorbar(im, ax=ax, label='Condições com A > 1 em pelo menos 2 réplicas (%)')
ax.set_title('Excedência por bloco nas condições completas')
save(fig, 'Figura_02_excedencia_blocos', 'Figura 2. Excedência das referências nos sete blocos, em pelo menos duas réplicas de desenvolvimento. Os denominadores são as condições completas de cada local (2.447, 2.456, 2.455 e 2.460). A excedência isolada não aplica a regra completa de prioridade nem constitui teste de significância. Fonte: resultados da pesquisa, tabelas locais 09_excedencia_blocos.')

## 3. Concordância das magnitudes entre locais

Usamos as mesmas 2.420 condições compartilhadas. A correlação compara a ordenação dos escores locais, não demonstra que os valores estejam na mesma escala biológica.

In [ ]:
d = pd.read_csv(T / 'concordancia_pares_e_placas.csv')
d = d[d.placa.eq('todas')]
a = np.eye(4)
for r in d.itertuples():
    i = SITES.index(r.site_a)
    j = SITES.index(r.site_b)
    a[i, j] = a[j, i] = r.Spearman_A_Mito_local
fig, ax = plt.subplots(figsize=(6, 4.3))
im = ax.imshow(a, vmin=0, vmax=1)
ax.set_xticks(range(4), SITES)
ax.set_yticks(range(4), SITES)
for i in range(4):
    for j in range(4):
        ax.text(j, i, f'{a[i, j]:.3f}', ha='center', va='center', bbox={'facecolor': 'white', 'alpha': 0.7, 'edgecolor': 'none', 'pad': 1})
fig.colorbar(im, ax=ax, label='Correlação de Spearman')
ax.set_title('Ordenação de A_Mito: 2.420 condições comuns')
save(fig, 'Figura_03_concordancia', 'Figura 3. Correlação de Spearman entre as medianas locais de A_Mito de 2.420 condições EOS–dose avaliáveis nos quatro locais. Características e referências foram estimadas localmente; a matriz compara ordenações de magnitudes, não perfis em uma representação comum ou transferência de modelo. Fonte: resultados da pesquisa, concordancia_pares_e_placas.csv.')

## 4. Erro de predição

Menor MAE indica menor erro absoluto médio na escala logarítmica. Os seis sentidos entre réplicas reutilizam compostos, e não são seis experimentos independentes.

In [ ]:
d = pd.read_csv(T / 'modelos_resumo.csv')
models = ['Mediana_treino', 'Copiar_escore', 'Calibracao_escore', 'Ridge_4_escores', 'Ridge_medidas', 'XGBoost']
labs = ['Mediana do treino', 'Cópia do escore', 'Calibração linear', 'Ridge: 4 escores', 'Ridge: medidas', 'XGBoost']
fig, ax = plt.subplots(figsize=(8.2, 4.8))
y = np.arange(6)
for j, site in enumerate(SITES):
    q = d[d.site.eq(site)].set_index('modelo').loc[models]
    ax.plot(q.MAE_log, y + (j - 1.5) * 0.12, marker=['o', 's', '^', 'D'][j], linestyle='none', label=site)
ax.set_yticks(y, labs)
ax.invert_yaxis()
ax.set_xlim(0, 0.37)
ax.set_xlabel('MAE em log(1 + A_Mito) — menor é melhor')
ax.set_title('Predições externas: os seis comparadores')
ax.legend(ncol=4, loc='lower right')
ax.grid(axis='x', alpha=0.2)
save(fig, 'Figura_04_modelos', 'Figura 4. Erro absoluto médio (MAE) na escala log(1+A_Mito), com peso igual às seis direções entre réplicas. Cada predição corresponde a códigos ausentes do treinamento do respectivo modelo. As direções reutilizam condições e não são experimentos independentes; a definição local do alvo antecedeu os folds. Fonte: resultados da pesquisa, modelos_resumo.csv.')

## 5. Contribuições SHAP por bloco

A soma absoluta depende de quantas medidas compõem cada bloco e de suas correlações. Estes percentuais explicam a predição e não o efeito biológico.

In [ ]:
d = pd.read_csv(T / 'SHAP_familias_blocos_quatro_locais.csv')
d['grupo'] = d.bloco.where(~d.bloco.str.startswith('Intercanais'), 'Intercanais')
g = d.groupby(['site', 'grupo']).soma_abs.sum().unstack(fill_value=0).loc[SITES]
g = g.div(g.sum(axis=1), axis=0) * 100
groups = ['Mito', 'AGP', 'DNA', 'ER', 'Morfologia_Cells', 'Morfologia_Cyto', 'Morfologia_Nuc', 'Intercanais']
fig, ax = plt.subplots(figsize=(8, 3.6))
bottom = np.zeros(4)
for col in groups:
    ax.barh(range(4), g[col], left=bottom, label=col.replace('Morfologia_', 'Forma '))
    bottom += g[col].values
ax.set_yticks(range(4), SITES)
ax.invert_yaxis()
ax.set_xlim(0, 100)
ax.set_xlabel('Participação na soma das médias de |SHAP| (%)')
ax.set_title('Contribuições preditivas por bloco')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.22), ncol=4, fontsize=8)
save(fig, 'Figura_05_SHAP', 'Figura 5. Participação dos blocos na soma das atribuições SHAP absolutas médias externas. Cada local reúne 30 modelos distintos; características ausentes em um ajuste recebem zero no resumo. As porcentagens dependem do número de medidas e de suas correlações; não são frações de efeito biológico ou contribuição causal. As medidas de Mito estavam entre os preditores. Fonte: resultados da pesquisa, SHAP_familias_blocos_quatro_locais.csv.')

## 6. Alda-1 e os 28 controles por placa e réplica

Cada ponto representa um perfil agregado de poço. A coordenada horizontal é o desvio normalizado por mediana/MAD (z), não a razão de brilho nem o escore A_Mito. Zero indica a mediana dos controles. Os dois controles efetivamente mostrados nas imagens são destacados, sem exclusão dos demais.

In [ ]:
v = pd.read_csv(A / 'intensidades_12_medidas_todos_DMSO.csv')
primary = v[v.medida.eq('Cells_Intensity_MeanIntensity_Mito')].copy()
refs = [(s, r) for s in ['IMTM', 'FMP', 'MEDINA', 'USC'] for r in ['R1', 'R2', 'R3']]
fig, ax = plt.subplots(figsize=(10, 7))
primary['y'] = [refs.index((s, r)) for s, r in zip(primary.site, primary.replica)]
dm = primary[primary.composto.eq('DMSO')]
ax.scatter(dm.z_MAD, dm.y, s=24, alpha=0.28, marker='o', label='28 DMSO')
for i, mk in [(1, 's'), (2, '^')]:
    d = primary[primary.papel_ficha.eq(f'DMSO {i}')]
    ax.scatter(d.z_MAD, d.y + 0.08 * (i * 2 - 3), s=56, marker=mk, label=f'DMSO {i} exibido')
d = primary[primary.composto.eq('EOS100315')]
ax.scatter(d.z_MAD, d.y, s=70, marker='D', label='Alda-1')
ax.axvline(0, linestyle=':', linewidth=1)
ax.set_yticks(range(12), [f'{s}  {r}' for s, r in refs])
ax.invert_yaxis()
ax.set_xlabel('MeanIntensity de Mito em Cells, normalizada por MAD local (z)')
ax.set_title('Alda-1 e os controles apresentados no contexto dos 28 DMSO')
ax.legend(loc='lower right', fontsize=8)
ax.spines[['top', 'right']].set_visible(False)
fig.text(0.01, 0.01, 'Um ponto = perfil agregado de um poço; não é brilho do campo inteiro.\nZero = mediana DMSO local; não é um limiar de atividade. Deslocamento vertical apenas para distinguir marcadores.', fontsize=8)
fig.tight_layout(rect=(0, 0.06, 1, 1))
fig.savefig(F / 'Figura_06_Alda1_controles.png', dpi=200)
fig.savefig(F / 'Figura_06_Alda1_controles.svg')
plt.close(fig)
CAP['Figura_06_Alda1_controles'] = 'Figura 6. Intensidade agregada de Mito em Cells de Alda-1 e dos 28 DMSO de B1001, por local e réplica, expressa em desvios normalizados pela mediana/MAD local (z). Os dois DMSO exibidos nos painéis estão destacados na distribuição completa. Zero corresponde à mediana dos controles e não a um limite de significância. A medida é a mediana por poço das médias de intensidade por objeto dos autores, não o brilho do campo inteiro. Fonte: integração dos perfis com a ficha visual humana.'

## 7. Preservar a leitura humana e o painel de microscopia

A tabela de 12 linhas é exportada sem alterar as respostas. A Figura 7 do manuscrito é um painel real, previamente produzido no USC/R1/campo 1; sua inclusão ilustra as janelas de exibição, não representa todos os campos ou réplicas. Os identificadores mascarados e os pixels são preservados.

In [ ]:
integracao = pd.read_csv(A / 'integracao_12_condicoes.csv')
if len(integracao) != 12 or integracao.duplicated(['site', 'replica']).any():
    raise ValueError('A tabela visual nao possui uma linha unica por local e replica.')
integracao.to_csv(F / 'Tabela_8_Alda1_12_avaliacoes.csv', index=False)
origem = BASE / 'fontes/imagens/USC_B1001_R1_campo1_Mito_duas_faixas_inteiro.png'
destino = F / 'Figura_07_exemplo_microscopia.png'
shutil.copy2(origem, destino)
assert hashlib.sha256(origem.read_bytes()).digest() == hashlib.sha256(destino.read_bytes()).digest()
CAP['Figura_07_exemplo_microscopia'] = 'Figura 7. Exemplo documental de visualização de Mito no USC, B1001/R1, campo 1, com os identificadores mascarados originais. As duas linhas mostram os mesmos campos: janela referenciada aos DMSO e janela complementar comum às três amostras. A ampliação recupera detalhes de exibição, não corrige iluminação nem modifica os TIFFs. Este único campo não é representativo de todas as réplicas; a síntese humana classificou USC como variável em R1–R3. Fonte: painéis reais da inspeção. Nenhuma escala física foi presumida.'
(F / 'legendas.json').write_text(json.dumps(CAP, ensure_ascii=False, indent=2), encoding='utf-8')
print('Concluido. Figuras e tabela:', F)
display(integracao[['site', 'replica', 'Mito_visual', 'A_Mito', 'razao_mediana_DMSO']])
